# Projeto RETAILL
<center><img src="./img/retaill.png" width="500" /></center>

## Retail Consult
<center><img src="./img/rc.svg" width="500" /></center>


In [3]:
import ipywidgets as widgets
from IPython.display import display

# =============================================================================
# SIMULADOR DE PÓS-COLHEITA (didático)
# -----------------------------------------------------------------------------
# O que modela:
#   1) FIRMEZA (amolecimento): modelo ODE
#         dF/dt = -k_F(t) * (F - F_min)
#      onde k_F(t) depende de temperatura (Arrhenius) e VPD (Défice de Pressão de Vapor).
#
#   3) BRIX (doçura): modelo ODE logístico (não “químico”, mas coerente)
#         dB/dt = r(t) * x * (1 - x/K)    com x = B - B_min e K = B_max - B_min
#
#   3) ACIDEZ: modelo ODE
#         dA/dt = -k_A(t) * (A - A_min)
#      onde k_A(t) depende de temperatura (Arrhenius).
#
#   4) TEMPO DE VIDA RESTANTE:
#      consumo acelerado por temperatura (Arrhenius), Thermal Time e VPD.
#
#   5) BOLOR/PODRIDÃO: estado mold(t) (0-1) que cresce quando VPD é baixo (alta humidade).
#
# IMPORTANTE:
#   - Unidades de "firmeza" são tipicamente em Newtons (N) medidas com penetrómetro.
#     Os valores aqui são calibrados para serem semelhantes a essa realidade.
#   - Os parâmetros por fruta são heurísticos e servem para simulação/ensino.
# =============================================================================


# =============================================================================
# 1) PRESETS POR FRUTA
# -----------------------------------------------------------------------------
# Cada fruta tem:
#
#   Termodinâmica / cinética:
#     - Tref_C      : temperatura de referência dos parâmetros (°C)
#     - Ea_J        : energia de ativação p/ amolecimento (J/mol)
#     - k_firm_ref  : taxa base de amolecimento a Tref (1/dia aprox)
#

#   Humidade:
#     - RH_ref      : RH "ideal" (ou típica de armazenamento) para essa fruta (%)
#     - beta_RH     : sensibilidade à RH baixa (desidratação) no amolecimento
#
#   Firmeza:
#     - firmeza_0_default : firmeza típica no dia 0 (valor inicial sugerido, em N)
#     - firmeza_min       : limite mínimo (assíntota) de amolecimento (em N)
#
#   Brix:
#     - brix_0_default : brix típico no dia 0
#     - brix_min/max   : limites mínimos/máximos teóricos no modelo
#     - brix_g         : taxa base logística (não é “dias até maturação”)
#
#   Qualidade (índice 0–100):
#     - qual_firm_threshold : “limiar” de firmeza considerado aceitável (em N)
#     - qual_brix_target    : brix "alvo" (pico do score de brix)
#
#   Acidez:
#     - acidez_0_default : acidez típica no dia 0
#     - acidez_min       : acidez mínima teórica
#     - k_acidez_ref     : taxa base de degradação da acidez
#     - Ea_acidez_J      : energia de ativação
#     - qual_acidez_target: alvo de acidez para máxima qualidade
#
#   Tempo de Vida (Shelf Life):
#     - SL_ref           : tempo de vida máximo de referência em dias
#   Bolor/podridão (RH alta):
#     - RH_mold_thr     : limiar (%) a partir do qual há risco significativo
#     - mold_rate_ref   : taxa base de crescimento do bolor em Tref (1/dia)
#     - mold_sens_RH    : sensibilidade ao excedente de RH acima do limiar
#     - mold_max_penalty: penalização máxima na qualidade (0..1)
#     - Ea_mold_J       : energia de ativação para crescimento de bolor
# =============================================================================

PRESETS = {
    # -------------------------------------------------------------------------
    # KIWI
    # -------------------------------------------------------------------------
    "kiwi_hayward": {
        "label": "Kiwi (Hayward)",
        "Tref_C": 5.0, "Ea_J": 60000, "k_firm_ref": 0.06, "beta_RH": 1.2, "RH_ref": 90,
        "firmeza_min": 2,  # N
        "firmeza_0_default": 18, # N
        "brix_min": 11, "brix_max": 17, "brix_g": 0.35,
        "brix_0_default": 11.0,
        "qual_firm_threshold": 5, "qual_brix_target": 15, "acidez_0_default": 1.5, "acidez_min": 0.5, "k_acidez_ref": 0.02, "Ea_acidez_J": 55000, "qual_acidez_target": 1.0, "SL_ref": 30,
    },
    "kiwi_baby": {
        "label": "Kiwi (Baby/Berry)",
        "Tref_C": 4.0, "Ea_J": 58000, "k_firm_ref": 0.14, "beta_RH": 2.0, "RH_ref": 95,
        "firmeza_min": 1.5, "firmeza_0_default": 40,
        "brix_min": 8.0, "brix_max": 18.0, "brix_g": 0.50, "brix_0_default": 8.0,
        "qual_firm_threshold": 6.0, "qual_brix_target": 17.0, "acidez_0_default": 1.1, "acidez_min": 0.5,
        "k_acidez_ref": 0.02, "Ea_acidez_J": 55000, "qual_acidez_target": 1.0, "SL_ref": 45,
    },

    # -------------------------------------------------------------------------
    # APPLE
    # -------------------------------------------------------------------------
    "maca_fuji": {
        "label": "Maçã (Fuji)",
        "Tref_C": 5.0, "Ea_J": 47000, "k_firm_ref": 0.018, "beta_RH": 0.7, "RH_ref": 90,
        "firmeza_min": 25.0, "firmeza_0_default": 85,
        "brix_min": 13.0, "brix_max": 17.0, "brix_g": 0.15, "brix_0_default": 13.0,
        "qual_firm_threshold": 45.0, "qual_brix_target": 16.0, "acidez_0_default": 0.4, "acidez_min": 0.5,
        "k_acidez_ref": 0.02, "Ea_acidez_J": 55000, "qual_acidez_target": 1.0, "SL_ref": 180,
    },
    "maca_golden": {
        "label": "Maçã (Golden)",
        "Tref_C": 5.0, "Ea_J": 50000, "k_firm_ref": 0.025, "beta_RH": 0.8, "RH_ref": 90,
        "firmeza_min": 20.0, "firmeza_0_default": 70,
        "brix_min": 11.0, "brix_max": 14.5, "brix_g": 0.18, "brix_0_default": 11.0,
        "qual_firm_threshold": 40.0, "qual_brix_target": 13.5, "acidez_0_default": 0.5, "acidez_min": 0.5,
        "k_acidez_ref": 0.02, "Ea_acidez_J": 55000, "qual_acidez_target": 1.0, "SL_ref": 120,
    },
    "maca_gala": {
        "label": "Maçã (Gala)",
        "Tref_C": 5.0, "Ea_J": 48000, "k_firm_ref": 0.040, "beta_RH": 0.9, "RH_ref": 90,
        "firmeza_min": 16.0, "firmeza_0_default": 70,
        "brix_min": 12.0, "brix_max": 15.0, "brix_g": 0.20, "brix_0_default": 12.0,
        "qual_firm_threshold": 30.0, "qual_brix_target": 14.5, "acidez_0_default": 0.4, "acidez_min": 0.5,
        "k_acidez_ref": 0.02, "Ea_acidez_J": 55000, "qual_acidez_target": 1.0, "SL_ref": 90,
    },
    "maca_reineta": {
        "label": "Maçã (Reineta)",
        "Tref_C": 5.0, "Ea_J": 52000, "k_firm_ref": 0.035, "beta_RH": 1.0, "RH_ref": 90,
        "firmeza_min": 18.0, "firmeza_0_default": 65,
        "brix_min": 10.5, "brix_max": 13.5, "brix_g": 0.16, "brix_0_default": 10.5,
        "qual_firm_threshold": 35.0, "qual_brix_target": 12.5, "acidez_0_default": 0.8, "acidez_min": 0.5,
        "k_acidez_ref": 0.02, "Ea_acidez_J": 55000, "qual_acidez_target": 1.0, "SL_ref": 90,
    },

    # -------------------------------------------------------------------------
    # BERRIES
    # -------------------------------------------------------------------------
    "morango": {
        "label": "Morango",
        "Tref_C": 2.0, "Ea_J": 52000, "k_firm_ref": 0.120, "beta_RH": 2.4, "RH_ref": 95,
        "firmeza_min": 1.0, "firmeza_0_default": 5,
        "brix_min": 7.5, "brix_max": 8.0, "brix_g": 0.01, "brix_0_default": 7.5,
        "qual_firm_threshold": 2.0, "qual_brix_target": 7.5, "acidez_0_default": 0.8, "acidez_min": 0.5,
        "k_acidez_ref": 0.02, "Ea_acidez_J": 55000, "qual_acidez_target": 1.0, "SL_ref": 7,
    },
    "framboesa": {
        "label": "Framboesa",
        "Tref_C": 2.0, "Ea_J": 56000, "k_firm_ref": 0.060, "beta_RH": 2.2, "RH_ref": 95,
        "firmeza_min": 1.5, "firmeza_0_default": 6,
        "brix_min": 9.5, "brix_max": 10.0, "brix_g": 0.01, "brix_0_default": 9.5,
        "qual_firm_threshold": 2.5, "qual_brix_target": 9.5, "acidez_0_default": 1.2, "acidez_min": 0.5,
        "k_acidez_ref": 0.02, "Ea_acidez_J": 55000, "qual_acidez_target": 1.0, "SL_ref": 7,
    },
    "mirtilo": {
        "label": "Mirtilo",
        "Tref_C": 2.0, "Ea_J": 52000, "k_firm_ref": 0.030, "beta_RH": 1.6, "RH_ref": 95,
        "firmeza_min": 3.0, "firmeza_0_default": 10,
        "brix_min": 11.5, "brix_max": 12.0, "brix_g": 0.01, "brix_0_default": 11.5,
        "qual_firm_threshold": 4.0, "qual_brix_target": 11.5, "acidez_0_default": 0.6, "acidez_min": 0.5,
        "k_acidez_ref": 0.02, "Ea_acidez_J": 55000, "qual_acidez_target": 1.0, "SL_ref": 21,
    },
    # -------------------------------------------------------------------------
    # POME
    # -------------------------------------------------------------------------
    "cereja": {
        "label": "Cereja",
        "Tref_C": 2.0, "Ea_J": 48000, "k_firm_ref": 0.045, "beta_RH": 1.6, "RH_ref": 95,
        "firmeza_min": 8.0, "firmeza_0_default": 15,
        "brix_min": 16.0, "brix_max": 16.5, "brix_g": 0.01, "brix_0_default": 16.0,
        "qual_firm_threshold": 10.0, "qual_brix_target": 16.0, "acidez_0_default": 0.5, "acidez_min": 0.5,
        "k_acidez_ref": 0.02, "Ea_acidez_J": 55000, "qual_acidez_target": 1.0, "SL_ref": 21,
    },
    "pessego": {
        "label": "Pêssego",
        "Tref_C": 2.0, "Ea_J": 56000, "k_firm_ref": 0.080, "beta_RH": 1.1, "RH_ref": 92,
        "firmeza_min": 4.0, "firmeza_0_default": 40,
        "brix_min": 10.0, "brix_max": 14.0, "brix_g": 0.20, "brix_0_default": 10.0,
        "qual_firm_threshold": 8.0, "qual_brix_target": 13.0, "acidez_0_default": 0.6, "acidez_min": 0.5,
        "k_acidez_ref": 0.02, "Ea_acidez_J": 55000, "qual_acidez_target": 1.0, "SL_ref": 14,
    },
    "ameixa": {
        "label": "Ameixa",
        "Tref_C": 2.0, "Ea_J": 52000, "k_firm_ref": 0.060, "beta_RH": 1.0, "RH_ref": 92,
        "firmeza_min": 5.0, "firmeza_0_default": 35,
        "brix_min": 10.0, "brix_max": 16.0, "brix_g": 0.25, "brix_0_default": 10.0,
        "qual_firm_threshold": 10.0, "qual_brix_target": 15.0, "acidez_0_default": 0.8, "acidez_min": 0.5,
        "k_acidez_ref": 0.02, "Ea_acidez_J": 55000, "qual_acidez_target": 1.0, "SL_ref": 21,
    },

    # -------------------------------------------------------------------------
    # CITRUS
    # -------------------------------------------------------------------------
    "laranja": {
        "label": "Laranja",
        "Tref_C": 5.0, "Ea_J": 42000, "k_firm_ref": 0.010, "beta_RH": 0.35, "RH_ref": 90,
        "firmeza_min": 20.0, "firmeza_0_default": 50,
        "brix_min": 11.0, "brix_max": 11.5, "brix_g": 0.01, "brix_0_default": 11.0,
        "qual_firm_threshold": 35.0, "qual_brix_target": 11.0, "acidez_0_default": 1.0, "acidez_min": 0.5,
        "k_acidez_ref": 0.02, "Ea_acidez_J": 55000, "qual_acidez_target": 1.0, "SL_ref": 90,
    },

    # -------------------------------------------------------------------------
    # OTHERS
    # -------------------------------------------------------------------------
    "banana": {
        "label": "Banana",
        "Tref_C": 14.0, "Ea_J": 65000, "k_firm_ref": 0.090, "beta_RH": 1.0, "RH_ref": 90,
        "firmeza_min": 5.0, "firmeza_0_default": 80,
        "brix_min": 5.0, "brix_max": 20.0, "brix_g": 0.50, "brix_0_default": 5.0,
        "qual_firm_threshold": 15.0, "qual_brix_target": 19.0, "acidez_0_default": 0.4, "acidez_min": 0.5,
        "k_acidez_ref": 0.02, "Ea_acidez_J": 55000, "qual_acidez_target": 1.0, "SL_ref": 21,
    },
    "pera": {
        "label": "Pera",
        "Tref_C": 2.0, "Ea_J": 54000, "k_firm_ref": 0.050, "beta_RH": 1.2, "RH_ref": 92,
        "firmeza_min": 6.0, "firmeza_0_default": 50,
        "brix_min": 11.0, "brix_max": 15.0, "brix_g": 0.25, "brix_0_default": 11.0,
        "qual_firm_threshold": 12.0, "qual_brix_target": 14.0, "acidez_0_default": 0.3, "acidez_min": 0.5,
        "k_acidez_ref": 0.02, "Ea_acidez_J": 55000, "qual_acidez_target": 1.0, "SL_ref": 90,
    },
    "uva": {
        "label": "Uva",
        "Tref_C": 2.0, "Ea_J": 45000, "k_firm_ref": 0.020, "beta_RH": 1.8, "RH_ref": 92,
        "firmeza_min": 5.0, "firmeza_0_default": 15,
        "brix_min": 16.0, "brix_max": 16.5, "brix_g": 0.01, "brix_0_default": 16.0,
        "qual_firm_threshold": 8.0, "qual_brix_target": 16.0, "acidez_0_default": 0.6, "acidez_min": 0.5,
        "k_acidez_ref": 0.02, "Ea_acidez_J": 55000, "qual_acidez_target": 1.0, "SL_ref": 45,
    },
    "figo": {
        "label": "Figo",
        "Tref_C": 2.0, "Ea_J": 52000, "k_firm_ref": 0.110, "beta_RH": 1.8, "RH_ref": 95,
        "firmeza_min": 1.0, "firmeza_0_default": 8,
        "brix_min": 16.0, "brix_max": 20.0, "brix_g": 0.15, "brix_0_default": 16.0,
        "qual_firm_threshold": 2.0, "qual_brix_target": 19.0, "acidez_0_default": 0.3, "acidez_min": 0.5,
        "k_acidez_ref": 0.02, "Ea_acidez_J": 55000, "qual_acidez_target": 1.0, "SL_ref": 7,
    },
    "melao": {
        "label": "Melão",
        "Tref_C": 7.0, "Ea_J": 52000, "k_firm_ref": 0.060, "beta_RH": 0.9, "RH_ref": 90,
        "firmeza_min": 5.0, "firmeza_0_default": 20,
        "brix_min": 10.0, "brix_max": 14.0, "brix_g": 0.22, "brix_0_default": 10.0,
        "qual_firm_threshold": 8.0, "qual_brix_target": 13.5, "acidez_0_default": 0.2, "acidez_min": 0.5,
        "k_acidez_ref": 0.02, "Ea_acidez_J": 55000, "qual_acidez_target": 1.0, "SL_ref": 21,
    },
}


MOLD_DEFAULTS = {
    "RH_mold_thr": 95.0,        # RH (%) a partir do qual começa risco significativo
    "mold_rate_ref": 0.06,      # taxa base (1/dia) em Tref
    "mold_sens_RH": 10.0,       # sensibilidade ao excedente de RH acima do limiar
    "mold_max_penalty": 0.80,   # máximo de penalização (0..1)
    "Ea_mold_J": 45000.0,       # sensibilidade à temperatura (Arrhenius)
}

MOLD_BY_FRUIT = {
    # Muito sensíveis
    "morango":   {"RH_mold_thr": 93.0, "mold_rate_ref": 0.22, "mold_sens_RH": 16.0, "mold_max_penalty": 0.95, "Ea_mold_J": 52000.0},
    "framboesa": {"RH_mold_thr": 93.0, "mold_rate_ref": 0.20, "mold_sens_RH": 16.0, "mold_max_penalty": 0.95, "Ea_mold_J": 52000.0},
    "figo":      {"RH_mold_thr": 93.0, "mold_rate_ref": 0.18, "mold_sens_RH": 14.0, "mold_max_penalty": 0.92, "Ea_mold_J": 50000.0},

    # Sensíveis
    "mirtilo":   {"RH_mold_thr": 94.0, "mold_rate_ref": 0.12, "mold_sens_RH": 14.0, "mold_max_penalty": 0.90, "Ea_mold_J": 48000.0},
    "cereja":    {"RH_mold_thr": 94.0, "mold_rate_ref": 0.10, "mold_sens_RH": 13.0, "mold_max_penalty": 0.85, "Ea_mold_J": 47000.0},
    "pessego":   {"RH_mold_thr": 94.0, "mold_rate_ref": 0.10, "mold_sens_RH": 12.0, "mold_max_penalty": 0.85, "Ea_mold_J": 48000.0},
    "ameixa":    {"RH_mold_thr": 94.0, "mold_rate_ref": 0.09, "mold_sens_RH": 12.0, "mold_max_penalty": 0.82, "Ea_mold_J": 47000.0},

    # Médio
    "pera":      {"RH_mold_thr": 95.0, "mold_rate_ref": 0.07, "mold_sens_RH": 10.0, "mold_max_penalty": 0.75, "Ea_mold_J": 45000.0},
    "melao":     {"RH_mold_thr": 95.0, "mold_rate_ref": 0.08, "mold_sens_RH": 10.0, "mold_max_penalty": 0.78, "Ea_mold_J": 45000.0},
    "uva":       {"RH_mold_thr": 95.0, "mold_rate_ref": 0.08, "mold_sens_RH": 11.0, "mold_max_penalty": 0.80, "Ea_mold_J": 45000.0},

    # Baixo a médio (kiwis/maçãs)
    "kiwi_hayward": {"RH_mold_thr": 95.0, "mold_rate_ref": 0.05,  "mold_sens_RH": 9.0,  "mold_max_penalty": 0.65, "Ea_mold_J": 43000.0},
    "kiwi_baby":    {"RH_mold_thr": 95.0, "mold_rate_ref": 0.07,  "mold_sens_RH": 10.0, "mold_max_penalty": 0.75, "Ea_mold_J": 45000.0},
    "maca_golden":  {"RH_mold_thr": 95.0, "mold_rate_ref": 0.04,  "mold_sens_RH": 8.0,  "mold_max_penalty": 0.60, "Ea_mold_J": 42000.0},
    "maca_reineta": {"RH_mold_thr": 95.0, "mold_rate_ref": 0.05,  "mold_sens_RH": 9.0,  "mold_max_penalty": 0.65, "Ea_mold_J": 43000.0},
    "maca_gala":    {"RH_mold_thr": 95.0, "mold_rate_ref": 0.05,  "mold_sens_RH": 9.0,  "mold_max_penalty": 0.65, "Ea_mold_J": 43000.0},
    "maca_fuji":    {"RH_mold_thr": 95.0, "mold_rate_ref": 0.035, "mold_sens_RH": 8.0,  "mold_max_penalty": 0.55, "Ea_mold_J": 42000.0},

    # Baixo (citrinos)
    "laranja": {"RH_mold_thr": 96.0, "mold_rate_ref": 0.03, "mold_sens_RH": 8.0, "mold_max_penalty": 0.50, "Ea_mold_J": 42000.0},
}

# Aplica defaults e depois overrides por fruta
for k in PRESETS:
    for kk, vv in MOLD_DEFAULTS.items():
        PRESETS[k].setdefault(kk, vv)
    if k in MOLD_BY_FRUIT:
        PRESETS[k].update(MOLD_BY_FRUIT[k])


# =============================================================================
# 3) UI (widgets)
# =============================================================================
fruit_widget = widgets.Dropdown(
    options=[(v["label"], k) for k, v in PRESETS.items()],
    value="kiwi_hayward",
    description="Fruta:",
)

days_widget = widgets.IntText(
    value=5,
    description=f'Dias',
    continuous_update=False,
)
#Temperatura 10 dias
temp_box = widgets.Box(layout=widgets.Layout(
    width='100%',
    display='flex',
    flex_wrap='wrap'
))
rh_box = widgets.Box(layout=widgets.Layout(
    width='100%',
    display='flex',
    flex_wrap='wrap'
))
temp_widgets = []
rh_widgets = []

# Valores iniciais (t=0) ajustáveis: mudam automaticamente com a fruta
firmeza0_widget = widgets.FloatText(description="Firmeza (dia 0):")
brix0_widget   = widgets.FloatText(description="°Brix (dia 0):")
acidity0_widget = widgets.FloatText(description="Acidez (dia 0):")

real_values_widget = widgets.Checkbox(
    value=False,
    description="Valores reais para comparação?",
    continuous_update=False
)
real_brix_box = widgets.Box(layout=widgets.Layout(
    width='100%',
    display='flex',
    flex_wrap='wrap'
))
real_firmness_box = widgets.Box(layout=widgets.Layout(
    width='100%',
    display='flex',
    flex_wrap='wrap'
))
real_acidity_box = widgets.Box(layout=widgets.Layout(
    width='100%',
    display='flex',
    flex_wrap='wrap'
))
real_firmness_widgets = []
real_brix_widgets = []
real_acidity_widgets = []

def update_ui(change):
    global temp_widgets, rh_widgets, real_firmness_widgets, real_brix_widgets
    days = int(change['new'])

    # --- Temperature & Humidity widgets ---
    temp_widgets.clear()
    rh_widgets.clear()
    temp_box.children = []
    rh_box.children = []

    for i in range(days):
        w = widgets.FloatText(
            value=12.0,
            description=f'T ºC Dia {i}',
            continuous_update=False,
            layout=widgets.Layout(width='400px'),
            style={'description_width': '100px'}
        )
        temp_widgets.append(w)

    for i in range(days):
        w = widgets.BoundedIntText(
            value=90,
            min=0,
            max=100,
            description=f'Hum R (%) Dia {i}',
            continuous_update=False,
            layout=widgets.Layout(width='400px'),
            style={'description_width': '100px'}
        )
        rh_widgets.append(w)

    temp_box.children = temp_widgets
    rh_box.children = rh_widgets

    # Real Firmness & Real Brix widgets
    real_firmness_widgets.clear()
    real_brix_widgets.clear()
    real_acidity_widgets.clear()

    if real_values_widget.value:
        for i in range(days):
            w1 = widgets.FloatText(
                value=0.0,
                description=f'Firm Dia {i}',
                continuous_update=False,
                layout=widgets.Layout(width='400px'),
                style={'description_width': '80px'}
            )
            real_firmness_widgets.append(w1)

            w2 = widgets.FloatText(
                value=0.0,
                description=f'Brix Dia {i}',
                continuous_update=False,
                layout=widgets.Layout(width='400px'),
                style={'description_width': '80px'}
            )
            real_brix_widgets.append(w2)

            w3 = widgets.FloatText(
                value=0.0,
                description=f'Acidez Dia {i}',
                continuous_update=False,
                layout=widgets.Layout(width='400px'),
                style={'description_width': '80px'}
            )
            real_acidity_widgets.append(w3)

        real_firmness_box.children = real_firmness_widgets
        real_brix_box.children = real_brix_widgets
        real_acidity_box.children = real_acidity_widgets
        real_firmness_box.layout.display = 'flex'
        real_brix_box.layout.display = 'flex'
        real_acidity_box.layout.display = 'flex'
    else:
        real_firmness_box.children = ()
        real_brix_box.children = ()
        real_acidity_box.children = ()
        real_firmness_box.layout.display = 'none'
        real_brix_box.layout.display = 'none'
        real_acidity_box.layout.display = 'none'

update_ui({'new': int(days_widget.value)})
days_widget.observe(update_ui, names='value')
real_values_widget.observe(lambda _: update_ui({'new': int(days_widget.value)}), names='value')

run_button = widgets.Button(description="Run simulation", button_style="success")

ui = widgets.VBox([
    widgets.HTML("<b>Fruta e dias:</b>"),
    fruit_widget,
    days_widget,
    widgets.HTML("<b>Temperatura (°C) por dia:</b>"),
    temp_box,
    widgets.HTML("<b>Humidade Relativa (%) por dia:</b>"),
    rh_box,
    widgets.HTML("<b>Valores iniciais típicos (ajustáveis) - t=0</b>"),
    firmeza0_widget,
    brix0_widget,
    acidity0_widget,
    widgets.HTML("<b>Valores Reais:</b>"),
    real_values_widget,
    real_firmness_box,
    real_brix_box,
    real_acidity_box,
    run_button
])

display(ui)

def set_initials_from_fruit(fruit_key: str):
    """Ao mudar de fruta no dropdown, atualiza os defaults de firmeza e brix no dia 0."""
    p = PRESETS[fruit_key]
    firmeza0_widget.value = float(p["firmeza_0_default"])
    brix0_widget.value   = float(p["brix_0_default"])
    acidity0_widget.value   = float(p["acidez_0_default"])

set_initials_from_fruit(fruit_widget.value)

def _on_fruit_changed(change):
    if change["name"] == "value":
        set_initials_from_fruit(change["new"])

fruit_widget.observe(_on_fruit_changed, names="value")


# =============================================================================
# 4) SIMULAÇÃO (Firmeza, Brix, Acidez, VPD, Shelf Life e Bolor)
# =============================================================================
def run_simulation(fruit_key: str, T_c: list[float], RH_pct: list[float], days: int,
                   firmeza_0_user: float, brix_0_user: float, acidez_0_user: float,
                   real_brix_values: list[float] | None = None, real_firmness_values: list[float] | None = None,
                   real_acidity_values: list[float] | None = None, dt: int | None = None, show_plots: bool = True
                   ) -> tuple[float, float, dict]:
    """
    Run simulation
    Args:
        fruit_key (str): The fruit key present on preset.
        T_c (list[float]): Temperature (in ºCelsius) Array.
        RH_pct (list[float]): Relative Humidity Array.
        days (int): The days for simulation.
        firmeza_0_user (float): The fruit firmness on day 0.
        brix_0_user (float): The fruit BRIX on day 0.
        acidez_0_user (float): The fruit acidity on day 0.
        real_brix_values (list[float] | None): No plot comparation to real BRIX values per day on None.
        real_firmness_values (list[float] | None): No plot comparation to real Firmness values per day on None.
        real_acidity_values (list[float] | None): No plot comparation to real Acidity values per day on None.
        dt (int | None): Time interval for simulation. 0.05 if None.
        show_plots (bool): If True, then show plots on runtime.

    Return:
        tuple[float, float, dict]: quality index on simulation last day, remaining lifetime on last day and info to plot.
    """
    import numpy as np
    import matplotlib.pyplot as plt
    dt = 0.05
    R = 8.314

    def k_temp_scaling(Ea, T, Tref):
        return np.exp((-Ea / R) * (1/T - 1/Tref))

    def calc_vpd(T_cels, RH_p):
        es = 0.6108 * np.exp(17.27 * T_cels / (T_cels + 237.3))
        ea = es * (RH_p / 100.0)
        return es - ea

    p = PRESETS[fruit_key]
    t = np.arange(0, days, dt)
    T_c = np.repeat(np.array(T_c), int(1/dt))
    RH_pct = np.repeat(np.array(RH_pct), int(1/dt))

    T_K = T_c + 273.15
    Tref_K = p["Tref_C"] + 273.15

    # Thermal Time
    TT = np.zeros_like(t)
    T_base = 0
    for i in range(1, len(t)):
        TT[i] = TT[i-1] + max(0, T_c[i-1] - T_base) * dt

    # VPD
    VPD = calc_vpd(T_c, RH_pct)
    VPD_ref = calc_vpd(p["Tref_C"], p["RH_ref"])

    # Firmeza
    kT_firm = p["k_firm_ref"] * k_temp_scaling(p["Ea_J"], T_K, Tref_K)
    firmeza = np.zeros_like(t)
    firmeza_min = float(p["firmeza_min"])
    firmeza[0] = max(firmeza_min + 1e-6, float(firmeza_0_user))

    # Brix
    brix = np.zeros_like(t)
    brix_min = float(p["brix_min"])
    brix_max = float(p["brix_max"])
    brix[0] = float(brix_0_user)
    r0 = float(p["brix_g"])
    rT = k_temp_scaling(52000, T_K, Tref_K)

    # Acidez
    acidez = np.zeros_like(t)
    acidez_min = float(p["acidez_min"])
    acidez[0] = max(acidez_min + 1e-6, float(acidez_0_user))
    kT_acidez = p["k_acidez_ref"] * k_temp_scaling(p["Ea_acidez_J"], T_K, Tref_K)

    # Remaining Shelf Life
    SL_ref = float(p.get("SL_ref", 30))
    consumed_SL = np.zeros_like(t)

    for i in range(1, len(t)):
        VPD_excess = max(0, VPD[i-1] - VPD_ref)

        # Firmeza ODE
        k_VPD_firm = 1 + p.get("beta_RH", 1.0) * VPD_excess
        dD = (-kT_firm[i-1] * k_VPD_firm * (firmeza[i-1] - firmeza_min)) * dt
        firmeza[i] = max(firmeza_min, firmeza[i-1] + dD)

        # Brix ODE
        r_VPD_brix = max(0, 1.0 - 0.2 * VPD_excess)
        r_brix = r0 * rT[i-1] * r_VPD_brix
        x = max(0.0, brix[i-1] - brix_min)
        K = max(1e-6, (brix_max - brix_min))
        db = (r_brix * x * (1.0 - x / K)) * dt
        brix[i] = min(brix_max, max(brix[i-1] + db, brix_min))

        # Acidez ODE
        dA = (-kT_acidez[i-1] * (acidez[i-1] - acidez_min)) * dt
        acidez[i] = max(acidez_min, acidez[i-1] + dA)

        # Shelf Life Consumption
        r_T_SL = k_temp_scaling(55000, T_K[i-1], Tref_K)
        r_VPD_SL = 1 + 0.5 * VPD_excess
        consumed_SL[i] = consumed_SL[i-1] + (r_T_SL * r_VPD_SL) * dt

    remaining_SL = np.maximum(0, SL_ref - consumed_SL)

    # Quality
    firm_score = 1 / (1 + np.exp(-0.35 * (firmeza - float(p["qual_firm_threshold"]))))
    brix_score = np.exp(-((brix - float(p["qual_brix_target"]))**2) / 2)
    acidez_score = np.exp(-((acidez - float(p.get("qual_acidez_target", 1.0)))**2) / 0.5)

    # Índice de Maturação
    indice_maturacao = brix / acidez
    # 1. Calcular o Rácio Alvo (ideal) baseado nos presets
    target_ratio = float(p["qual_brix_target"]) / float(p.get("qual_acidez_target", 1.0))

    # 2. Criar um score para o rácio (penaliza desvios do rácio ideal)
    # O divisor 25.0 controla a "largura" da aceitação. Podes ajustar se quiseres ser mais rigoroso.
    ratio_score = np.exp(-((indice_maturacao - target_ratio)**2) / 25.0)

    quality_base = 100 * (0.40 * firm_score + 0.30 * ratio_score + 0.15 * brix_score + 0.15 * acidez_score)

    # Mold
    RH_mold_thr = float(p["RH_mold_thr"])
    mold_rate_ref = float(p["mold_rate_ref"])
    mold_sens_RH = float(p["mold_sens_RH"])
    mold_max_penalty = float(p["mold_max_penalty"])
    Ea_mold_J = float(p["Ea_mold_J"])

    mold_T = k_temp_scaling(Ea_mold_J, T_K, Tref_K)

    mold = np.zeros_like(t)
    mold[0] = 0.0
    for i in range(1, len(t)):
        VPD_thr = calc_vpd(T_c[i-1], RH_mold_thr)
        VPD_deficit = max(VPD_thr - VPD[i-1], 0) # Menor VPD = Maior humidade
        VPD_factor = 1.0 - np.exp(-mold_sens_RH * VPD_deficit * 5.0)
        rate = mold_rate_ref * mold_T[i-1] * VPD_factor
        dm = (rate * (1.0 - mold[i-1])) * dt
        mold[i] = min(1.0, np.max(np.append(np.array(mold[i-1] + dm), 0)))

    mold_penalty = mold_max_penalty * mold
    quality = quality_base * (1.0 - mold_penalty)

    # -------------------------------------------------------------------------
    # 4.8) Gráficos
    # -------------------------------------------------------------------------
    if show_plots:
        import matplotlib.pyplot as plt

        plt.figure(figsize=(10,4))
        plt.plot(t, firmeza, label="Firmeza", color="blue")
        plt.plot(t, brix, label="°Brix", color="orange")
        plt.plot(t, acidez * 10, label="Acidez (x10 para escala)", color="green")
        if(real_firmness_values is not None):
            plt.plot(np.arange(0, days, 1), np.array(real_firmness_values), 'o', label="Real Firmness", color="blue")
        if(real_brix_values is not None):
            plt.plot(np.arange(0, days, 1), np.array(real_brix_values), 'o', label="Real °Brix", color="orange")
        if(real_acidity_values is not None):
            plt.plot(np.arange(0, days, 1), np.array(real_acidity_values) * 10, 'o', label="Real Acidity", color="green")
        plt.legend()
        plt.xlabel("Dias")
        plt.title(f"{p['label']} - Evolução Físico-Química")
        plt.grid()
        plt.show()

        plt.figure(figsize=(10,4))
        plt.plot(t, indice_maturacao, label="Índice de Maturação", color="teal", linewidth=2)
        plt.ylabel("Rácio (Brix / Acidez)")
        plt.xlabel("Dias")
        plt.title("Evolução do Índice de Maturação")
        plt.grid(True)
        plt.legend()
        plt.show()

        plt.figure(figsize=(10,4))
        plt.plot(t, quality_base, label="Qualidade (base)")
        plt.plot(t, quality, linestyle='dashed', label="Qualidade (com bolor)")
        plt.ylabel("Qualidade (0-100)")
        plt.xlabel("Dias")
        plt.title("Índice de Qualidade")
        plt.grid()
        plt.legend()
        plt.show()

        plt.figure(figsize=(10,4))
        plt.plot(t, remaining_SL, label="Tempo de Vida Restante", color="red")
        plt.ylabel("Dias Restantes")
        plt.xlabel("Dias de Armazenamento")
        plt.title("Tempo de Vida Restante (Arrhenius, Thermal Time e VPD)")
        plt.grid()
        plt.legend()
        plt.show()

        plt.figure(figsize=(10,4))
        plt.plot(t, mold, label="Risco de bolor (0-1)")
        plt.plot(t, mold_penalty, label="Penalização aplicada (0-1)")
        plt.xlabel("Dias")
        plt.title("Bolor/podridão (Conduzido por VPD baixo)")
        plt.grid()
        plt.legend()
        plt.show()


        plt.figure(figsize=(10,4))
        plt.plot(t, VPD, label="VPD (kPa)", color="purple")
        plt.plot(t, TT / max(1, np.max(TT)) * np.max(VPD), label="Thermal Time (normalizado)", color="orange", linestyle="--")
        plt.xlabel("Dias")
        plt.title("Vapor Pressure Deficit (VPD) e Thermal Time")
        plt.grid()
        plt.legend()
        plt.show()
    try: 
        ra = (np.array(real_acidity_values) * 10).tolist()
    except: 
        ra = None
    plots_info = {
        "fq_evol": {
            "t": t.tolist(),
            "firmness": firmeza.tolist(),
            "brix": brix.tolist(),
            "acidity": (acidez * 10).tolist(),
            "real_t": np.arange(0, days, 1).tolist(),
            "real_firmness": np.array(real_firmness_values).tolist(),
            "real_brix": np.array(real_brix_values).tolist(),
            "real_acidity": ra,
        },
        "maturation": {
            "t": t.tolist(),
            "maturation": indice_maturacao.tolist(),
        },
        "quality": {
            "t": t.tolist(),
            "quality": quality.tolist(),
            "quality_base": quality_base.tolist()
        },
        "remaining_life": {
            "t": t.tolist(),
            "remaining_life": remaining_SL.tolist()
        },
        "mold": {
            "t": t.tolist(),
            "mold": mold.tolist(),
            "mold_penalty": mold_penalty.tolist(),
        },
        "vpd_tt": {
            "t": t.tolist(),
            "vpd": VPD.tolist(),
            "tt": (TT / max(1, np.max(TT)) * np.max(VPD)).tolist(),
        }

    }
    return quality[len(quality) - 1], remaining_SL[len(remaining_SL) - 1], plots_info


def on_run_clicked(_):
    run_simulation(
        fruit_widget.value,
        [temp_widget.value for temp_widget in temp_widgets],
        [rh_widget.value for rh_widget in rh_widgets],
        days_widget.value,
        firmeza0_widget.value,
        brix0_widget.value,
        acidity0_widget.value,
        real_brix_values=None if real_values_widget.value == False else  [real_brix_widget.value for real_brix_widget in real_brix_widgets],
        real_firmness_values=None if real_values_widget.value == False else [real_firmness_widget.value for real_firmness_widget in real_firmness_widgets],
        real_acidity_values=None if real_values_widget.value == False else [real_acidity_widget.value for real_acidity_widget in real_acidity_widgets]
    )

run_button.on_click(on_run_clicked)

In [ ]:
# FORECAST API ENDPOINT
# ========================================================================
import hashlib
from fastapi.responses import JSONResponse
import threading
from typing import List, Literal, Optional
from fastapi import FastAPI
from pydantic import BaseModel
import uvicorn
import uuid

app = FastAPI()

_inference_lock = threading.Lock()
_status_lock = threading.Lock()
_current_operation = {
    "operation": None,
    "client_id": None,
    "store_id": None,
    "product_id": None,
    "algorithm": None,
}

class HistoryEntry(BaseModel):
    """Single day of temperature and humidity data."""
    temperature: float          # Temperature in °C
    relative_humidity: float    # Relative humidity in % (0-100)
    real_brix_value: Optional[float] = None
    real_firmness_value: Optional[float] = None
    real_acidity_value: Optional[float] = None

class ForecastRequest(BaseModel):
    """Request schema for the /forecast endpoint."""
    service: Literal["inference"]
    client_id: str
    store_id: str
    algorithm: Literal["ode"]

    product_id: str
    history: List[HistoryEntry]
    horizon_days: int
    firmness_0_user: Optional[float] 
    brix_0_user: Optional[float]
    acidity_0_user: Optional[float]

class ForecastResponse(BaseModel):
    """Response schema for the /forecast endpoint."""
    service: Literal["inference"]
    client_id: str
    store_id: str
    algorithm: Literal["ode"]

    product_id: str
    quality_index: float
    remaining_lifetime: float
    plot_info: dict

class StatusResponse(BaseModel):
    """Response schema for the /status endpoint."""
    busy: bool
    operation: Optional[str] = None
    message: Literal["Available", "Busy"]
    client_id: Optional[str] = None
    store_id: Optional[str] = None
    product_id: Optional[str] = None
    algorithm: Optional[Literal["ode"]] = None

@app.post("/forecast", response_model=ForecastResponse)
def forecast(request: ForecastRequest):
    """
    POST endpoint to run a fruit quality forecast based on historical
    temperature and humidity data.
    
    Returns 429 if an inference is already running.
    """
    # --- Validation ---
    if request.service != "inference":
        return JSONResponse(
            status_code=400,
            content={"detail": "Only 'inference' service is supported."}
        )
    if request.algorithm != "ode":
        return JSONResponse(
            status_code=400,
            content={"detail": "Only 'ode' algorithm is supported."}
        )
    if request.product_id not in PRESETS:
        return JSONResponse(
            status_code=400,
            content={"detail": f"Unknown product_id: {request.product_id}. Valid keys: {list(PRESETS.keys())}"}
        )
    if len(request.history) != request.horizon_days:
        return JSONResponse(
            status_code=400,
            content={"detail": "history length must match horizon_days."}
        )

    # --- Rate limiting: check for concurrent inference on same product_id ---
    acquired = _inference_lock.acquire(blocking=False)
    operation_id = str(uuid.uuid4())
    with _status_lock:
        _current_operation.update({
            "operation": operation_id,
            "client_id": request.client_id,
            "store_id": request.store_id,
            "product_id": request.product_id,
            "algorithm": request.algorithm,
        })
    if not acquired:
        return JSONResponse(
            status_code=429,
            content={
                "detail": "Too Many Requests",
                "message": f"Inference already running for product_id"
            }
        )

    try:
        # --- Extract history arrays ---
        T_c = [h.temperature for h in request.history]
        RH_pct = [h.relative_humidity for h in request.history]
        rbv = [h.real_brix_value for h in request.history]
        rfv = [h.real_firmness_value for h in request.history]
        rav = [h.real_acidity_value for h in request.history]

        # --- Get preset defaults for initial values ---
        preset = PRESETS[request.product_id]
        firmeza_0 = preset["firmeza_0_default"] if request.firmness_0_user is None else request.firmness_0_user
        brix_0 = preset["brix_0_default"] if request.brix_0_user is None else request.brix_0_user
        acidez_0 = preset["acidez_0_default"] if request.acidity_0_user is None else request.acidity_0_user

        # --- Run simulation ---
        quality_index, lifetime, plot_info = run_simulation(
            fruit_key=request.product_id,
            T_c=T_c,
            RH_pct=RH_pct,
            days=request.horizon_days,
            firmeza_0_user=firmeza_0,
            brix_0_user=brix_0,
            acidez_0_user=acidez_0,
            real_brix_values=rbv,
            real_firmness_values=rfv,
            real_acidity_values=rav,
            show_plots=False
        )

        return ForecastResponse(
            service=request.service,
            client_id=request.client_id,
            store_id=request.store_id,
            algorithm=request.algorithm,
            product_id=request.product_id,
            quality_index=quality_index,
            remaining_lifetime=lifetime,
            plot_info=plot_info
        )
    finally:
        with _status_lock:
            _current_operation.update({
                "operation": None,
                "client_id": None,
                "store_id": None,
                "product_id": None,
                "algorithm": None,
            })
        _inference_lock.release()

@app.get("/status", response_model=StatusResponse)
def status():
    """
    GET endpoint that reports whether an inference is currently running.
    Takes no query params, path params, or special headers.
    """
    busy = _inference_lock.locked()

    if not busy:
        return StatusResponse(
            busy=False,
            operation=None,
            message="Available",
            client_id=None,
            store_id=None,
            product_id=None,
            algorithm=None,
        )

    with _status_lock:
        op = dict(_current_operation)

    return StatusResponse(
        busy=True,
        operation=op["operation"],
        message="Busy",
        client_id=op["client_id"],
        store_id=op["store_id"],
        product_id=op["product_id"],
        algorithm=op["algorithm"],
    )

def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8080)


server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

INFO:     Started server process [27200]
INFO:     Waiting for application startup.
INFO:     Application startup complete.


ERROR:    [Errno 10048] error while attempting to bind on address ('127.0.0.1', 8080): [winerror 10048] only one usage of each socket address (protocol/network address/port) is normally permitted
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


INFO:     127.0.0.1:55877 - "POST /forecast HTTP/1.1" 200 OK
INFO:     127.0.0.1:57591 - "POST /forecast HTTP/1.1" 200 OK
INFO:     127.0.0.1:61487 - "POST /forecast HTTP/1.1" 200 OK
INFO:     127.0.0.1:52019 - "POST /forecast HTTP/1.1" 200 OK
INFO:     127.0.0.1:55712 - "POST /forecast HTTP/1.1" 200 OK
